In [ ]:
#@title 💬 힌트 채팅 — 학번을 입력하고 이 셀을 실행하세요 { display-mode: "form" }
STUDENT_ID = "" #@param {type:"string"}

PROXY_URL = "https://script.google.com/macros/s/AKfycbwqvOFpghowbLkjAD49JOwutvKeDLbZmNZHHKxm0ciT6yU6LuDDcpCJ3rZbgiRwvoER/exec"

import html
import requests
import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import display

STUDENT_ID = STUDENT_ID.strip()
_ip = get_ipython()
_CHAT_CELL_MARK = "#@title 💬 힌트 채팅"


def _post(payload):
    try:
        res = requests.post(PROXY_URL, json=payload, timeout=60)
    except Exception as e:
        return None, f"서버에 연결하지 못했습니다: {e}"
    try:
        data = res.json()
    except ValueError:
        return None, f"서버 응답을 읽지 못했습니다 (HTTP {res.status_code}). 선생님께 알려주세요."
    if "error" in data:
        return None, data["error"]
    return data, None


def _request_hint(code="", error="", question=""):
    if not STUDENT_ID:
        return None, "학번을 입력하고 이 셀을 다시 실행하세요."
    return _post({"student_id": STUDENT_ID, "code": code, "error": error, "question": question})


def _bubble_html(text, mine):
    align = "flex-end" if mine else "flex-start"
    bg = "#dbeafe" if mine else "#f3f4f6"
    body = html.escape(text).replace("\n", "<br>")
    return (
        f'<div style="display:flex;justify-content:{align};margin:4px 0">'
        f'<div style="max-width:80%;padding:8px 12px;border-radius:12px;'
        f'background:{bg};color:#111;line-height:1.5;white-space:normal">{body}</div></div>'
    )


_status = widgets.HTML()
_tip = widgets.HTML(
    "<span style='color:#888;font-size:12px'>질문 1번에 1개씩 줄고, 매주 월요일에 다시 채워져요. "
    "셀을 실행하면 그 아래에 💬 이 셀 질문하기 버튼이, 에러가 나면 💡 힌트 요청 버튼이 나타나요.</span>")


def _show_status(remaining=None, paused=False):
    text = f"학번 <b>{html.escape(STUDENT_ID)}</b>"
    if remaining is not None:
        text += f" · 남은 질문 <b>{remaining}</b>개"
    if paused:
        text += " <span style='color:#b91c1c'>(지금은 사용 중지 상태예요)</span>"
    _status.value = f"<span style='color:#333'>{text}</span>"


def _show_error(message):
    _status.value = f"<span style='color:#b91c1c'>⚠️ {html.escape(message)}</span>"


def _refresh_status():
    data, err = _post({"student_id": STUDENT_ID, "action": "status"})
    if err:
        _show_error(err)
    else:
        _show_status(data["remaining"], data.get("paused", False))


_history = widgets.VBox(layout=widgets.Layout(
    max_height="360px", overflow_y="auto", border="1px solid #ddd", padding="8px"))
_input = widgets.Textarea(
    placeholder="막힌 부분이나 궁금한 점을 적어보세요. 코드는 필요한 부분만 붙여넣으세요.",
    layout=widgets.Layout(width="100%", height="80px"))
_send = widgets.Button(description="보내기", button_style="primary")


def _on_send(_):
    question = _input.value.strip()
    if not question:
        return
    _input.value = ""
    _send.disabled = True
    _send.description = "답변 기다리는 중..."
    _history.children += (widgets.HTML(_bubble_html(question, True)),)
    data, err = _request_hint(question=question)
    if err:
        _history.children += (widgets.HTML(_bubble_html("⚠️ " + err, False)),)
    else:
        _history.children += (widgets.HTML(_bubble_html(data["hint"], False)),)
        _show_status(data["remaining"])
    _send.disabled = False
    _send.description = "보내기"


_send.on_click(_on_send)

if STUDENT_ID:
    _show_status()
    _refresh_status()
else:
    _show_error("위 칸에 학번을 입력하고 이 셀을 다시 실행하세요. 학번을 넣어야 힌트 버튼도 나타납니다.")

display(widgets.VBox([_status, _tip, _history, _input, _send]))


def _error_hint_box(code_text, error_text):
    button = widgets.Button(description="💡 힌트 요청", button_style="info",
                            layout=widgets.Layout(width="140px"))
    answer = widgets.HTML()

    def _click(b):
        b.disabled = True
        b.description = "요청 중..."
        data, e = _request_hint(code=code_text, error=error_text)
        if e:
            answer.value = _bubble_html("⚠️ " + e, False)
            b.disabled = False
            b.description = "💡 힌트 요청"
        else:
            answer.value = _bubble_html(data["hint"], False)
            b.description = "힌트 받음"
            _show_status(data["remaining"])

    button.on_click(_click)
    return widgets.VBox([button, answer])


def _ask_box(code_text):
    toggle = widgets.Button(description="💬 이 셀 질문하기", layout=widgets.Layout(width="150px"))
    question_input = widgets.Text(placeholder="이 셀에서 궁금한 점을 적어보세요",
                                  layout=widgets.Layout(width="70%"))
    send = widgets.Button(description="보내기", button_style="primary",
                          layout=widgets.Layout(width="110px"))
    form = widgets.HBox([question_input, send], layout=widgets.Layout(display="none"))
    answers = widgets.VBox()

    def _toggle(_):
        form.layout.display = "flex" if form.layout.display == "none" else "none"

    def _send(_):
        question = question_input.value.strip()
        if not question:
            return
        send.disabled = True
        send.description = "기다리는 중..."
        data, e = _request_hint(code=code_text, question=question)
        if e:
            answers.children += (widgets.HTML(_bubble_html("⚠️ " + e, False)),)
        else:
            question_input.value = ""
            answers.children += (widgets.HTML(_bubble_html(question, True)),
                                 widgets.HTML(_bubble_html(data["hint"], False)))
            _show_status(data["remaining"])
        send.disabled = False
        send.description = "보내기"

    toggle.on_click(_toggle)
    send.on_click(_send)
    return widgets.VBox([toggle, form, answers])


def _on_cell_finished(result):
    code_text = result.info.raw_cell
    if not code_text.strip() or code_text.lstrip().startswith(_CHAT_CELL_MARK):
        return
    err = result.error_in_exec or result.error_before_exec
    if not STUDENT_ID:
        if err is not None:
            display(widgets.HTML(
                "<span style='color:#b91c1c'>💡 힌트를 받으려면 맨 위 '힌트 채팅' 셀에 학번을 입력하고 그 셀을 다시 실행하세요.</span>"))
        return
    if err is not None:
        display(_error_hint_box(code_text, f"{type(err).__name__}: {err}"))
    else:
        display(_ask_box(code_text))


if "_hint_hook" in globals():
    try:
        _ip.events.unregister("post_run_cell", _hint_hook)
    except ValueError:
        pass
_hint_hook = _on_cell_finished
_ip.events.register("post_run_cell", _hint_hook)

## 오늘의 실습
막히면 맨 위 **힌트 채팅**에 물어보세요. 에러가 난 셀 아래에는 **💡 힌트 요청** 버튼이 자동으로 나타납니다.
(질문 1번 = 토큰 1개)

# 딥러닝으로 손 글씨 분류하기 (교과서 96~101쪽)

**Ⅱ. 인공지능과 학습 · 2. 딥러닝 · 2-2-1 (2차시)**

MNIST 손 글씨 숫자(0~9) 이미지를 딥러닝으로 분류하는 모델을 만들고, 기계학습(의사결정트리)과 성능을 비교해 보자.

### 시작하기 전에
1. 이번에는 **파일 업로드가 없다.** 데이터는 케라스가 인터넷에서 직접 내려받는다.
2. 셀을 **위에서부터 순서대로** 실행한다 (Ctrl + Enter). `________` 빈칸은 스스로 학습 페이지를 보며 채운 뒤 실행한다.
3. ⑨번 셀은 **1분 정도** 걸린다. 끝날 때까지 기다린다.

| 번호 | 내용 | 교과서 |
|---|---|---|
| ① | 데이터셋 불러오기 | 96쪽 |
| ② | 데이터 형태 확인하기 | 97쪽 |
| ③ | 훈련 데이터 레이블별 개수 | 97쪽 |
| ④ | 전처리 — 차원 변환 | 98쪽 |
| ⑤ | 전처리 — 정규화 | 98쪽 |
| ⑥ | 전처리 — 원-핫 인코딩 | 98쪽 |
| ⑦ | 딥러닝 모델 구성 | 99쪽 |
| ⑧ | 학습 방법 정하기 | 99쪽 |
| ⑨ | 학습 | 100쪽 |
| ⑩ | 성능 평가 — 그래프 | 100쪽 |
| ⑪ | 딥러닝 정확도 | 101쪽 |
| ⑫ | 기계학습 정확도 | 101쪽 |


## ① 데이터셋 불러오기 (96쪽)
딥러닝 모듈 **케라스(keras)** 에서 MNIST 데이터셋을 불러온다. 훈련 데이터와 테스트 데이터가 나뉘어 돌아온다.

In [ ]:
# ① MNIST 데이터셋 불러오기
# [빈칸] 딥러닝 모듈 이름과, 데이터를 내려받는 함수 이름을 쓰세요.
from tensorflow import _____   # 딥러닝 모델 구현을 위한 모듈
mnist = keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist._________( )
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

> **확인** : `(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)` — 28×28 이미지가 훈련 6만 장, 테스트 1만 장

## ② 데이터 형태 확인하기 (97쪽)
이미지 10개를 레이블(정답)과 함께 그려 본다.
- `imshow( )` : 이미지를 그리는 함수. `cmap = 'Greys'` 는 회색조
- `set_title( )` : 그림 제목에 정답 숫자를 쓴다
- `subplots(1, 10)` 의 10과 `range(10)` 의 10은 **같아야 한다**. 칸이 10개이므로 반복도 10번. 다르면 칸이 비거나 없는 칸을 찾아 오류가 난다. 20개를 보려면 둘 다 20으로 바꾼다

In [ ]:
# ② 이미지 10개를 레이블과 함께 출력하기
# [빈칸] 이미지를 그리는 함수 이름과, 제목에 넣을 정답(레이블)을 쓰세요.
!pip install koreanize-matplotlib   # 한글 출력 그래프 모듈 설치
import koreanize_matplotlib         # 한글 출력 가능한 그래프 모듈 불러오기
import matplotlib.pyplot as plt     # 그래프 모듈
fig, axes = plt.subplots(1, 10, figsize = (5, 2))   # 1행 10열의 그래프
for i in range(10):                 # 10개의 이미지 데이터 출력
  image = X_train[i]
  axes[i].______(image, cmap = 'Greys')   # 그레이스케일 이미지 출력
  axes[i].set_xticks([])            # X축 출력하지 않기
  axes[i].set_yticks([])            # Y축 출력하지 않기
  axes[i].set_title(__________)   # 제목 = 레이블 번호
plt.show( )

> **확인** : 5, 0, 4, 1, 9, 2, 1, 3, 1, 4 손 글씨가 정답 숫자와 함께 나온다.

## ③ 훈련 데이터 레이블별 개수 (97쪽)
데이터가 레이블(0~9)별로 몇 개씩 있는지 세어 **편향**되어 있지 않은지 확인한다.
- `np.unique( )` : 중복되지 않는 고유한 값을 찾는다. `return_counts = True` 면 개수도 돌려준다

In [ ]:
# ③ 레이블별 개수 막대 그래프
# [빈칸] 고유한 값을 찾는 함수 이름과, 개수까지 돌려받는 옵션 값을 쓰세요.
import numpy as np
label_value, label_count = np.______(y_train, return_counts = ____)
plt.bar(label_value, label_count)   # 막대 그래프 그리기
plt.xticks(label_value)             # 모든 X축의 값 출력
plt.show( )

> **확인** : 레이블별로 5,000~7,000개씩 → 큰 차이가 없다.

## ④ 데이터 전처리 — 차원 변환 (98쪽)
이미지는 2차원(28×28)이지만 이 모델은 **1차원**으로 입력받는다. 한 줄(784개)로 펼친다.
- `reshape(60000, -1)` : 행은 60000, 나머지(-1)는 **자동 계산** → 784

In [ ]:
# ④ 2차원 이미지를 1차원으로 펼치기
# [빈칸] 배열 모양을 바꾸는 함수 이름과, 나머지 차원을 자동 계산하게 하는 값을 쓰세요.
print(X_train.shape, X_test.shape)          # 데이터의 형태 출력
X_train_flatten = X_train._______(60000, ____)   # 행의 크기를 60000으로 설정
X_test_flatten = X_test.reshape(10000, -1)    # 행의 크기를 10000으로 설정
print(X_train_flatten.shape, X_test_flatten.shape)

> **확인** : `(60000, 784) (10000, 784)` — 28 × 28 = 784

## ⑤ 데이터 전처리 — 정규화 (98쪽)
픽셀 값(0~255)을 최댓값으로 나눠 **0~1 사이**로 바꾼다. 편향성을 줄이고 학습 속도를 높인다.

In [ ]:
# ⑤ 0~255를 0~1로 정규화하기
# [빈칸] 픽셀 값의 최댓값을 쓰세요.
X_train_scaler = X_train_flatten.astype('float32') / ____
X_test_scaler = X_test_flatten.astype('float32') / 255   # 'float32'로 변환
X_train_scaler[0]   # 첫 번째 이미지 데이터 출력

> **확인** : 0~1 사이의 소수(float32)로 바뀐다.

## ⑥ 데이터 전처리 — 원-핫 인코딩 (98쪽)
레이블(0~9)을 0과 1로 된 길이 10짜리 벡터로 바꾼다. (93쪽 붓꽃의 (0, 0, 1)과 같은 원리)

In [ ]:
# ⑥ 레이블 원-핫 인코딩
# [빈칸] 레이블을 원-핫 인코딩하는 함수 이름을 쓰세요.
from keras.utils import ____________
y_train_hot = to_categorical(y_train)
y_test_hot = to_categorical(y_test)
y_train_hot[:2]   # 원-핫 인코딩한 결과 중 처음 두 개를 출력

> **확인** : `[0,0,0,0,0,1,0,0,0,0]` → 첫 이미지의 정답은 5, `[1,0,…]` → 두 번째는 0

## ⑦ 딥러닝 모델 구성 (99쪽)
은닉층이 **2개**인 모델. 입력층 = 특징 개수(784), 출력층 = 레이블 종류(10).
- 은닉층 활성화 함수 **렐루**, 출력층(다중 분류) **소프트맥스** (92쪽 표Ⅱ-9)

In [ ]:
# ⑦ 은닉층 2개짜리 딥러닝 모델 만들기
# [빈칸] 은닉층·출력층의 활성화 함수, 입력 특징 수, 두 번째 은닉층 노드 수, 출력 노드 수를 쓰세요.
from keras.models import Sequential   # 신경망 구축을 위한 모델
from keras.layers import Dense        # 완전 연결 레이어 정의를 위함
import tensorflow as tf               # 텐서플로 라이브러리 불러오기
tf.keras.utils.set_random_seed(1)     # 동일한 결과 재현을 위함
tf.config.experimental.enable_op_determinism( )
features = X_train_scaler.shape[1]    # 입력 데이터 특징의 수 (784)
labels = y_train_hot.shape[1]         # 출력 레이블의 개수 (10)
model = Sequential( )                 # Sequential 모델 생성
model.add(Dense(100, activation = '____', input_dim = ________))   # 은닉층 1
model.add(Dense(____, activation = 'relu'))                 # 은닉층 2
model.add(Dense(______, activation = '_______'))      # 출력층
model.summary( )                      # 모델 형태 출력

> **확인** : dense 78500, dense_1 5050, dense_2 510 → Total params 84,060 (Model 이름 뒤 번호는 달라도 된다)

## ⑧ 학습 방법 정하기 (99쪽)
손실함수는 다중 분류용 **범주형 교차 엔트로피**, 평가 지표는 **정확도** (94쪽 표Ⅱ-10)

In [ ]:
# ⑧ 손실함수와 평가 지표 정하기
# [빈칸] 다중 분류의 손실함수 이름과 평가 지표 이름을 쓰세요.
model.compile(loss = '____________', metrics = ['________'])

> **확인** : 출력 없이 실행되면 정상

## ⑨ 학습 (100쪽)
훈련 데이터로 **10번 반복(epochs)** 학습한다. 검증 데이터(validation_data)로 에포크마다 성능을 확인한다.
- 1분 정도 걸린다. **한 번만** 실행한다.

In [ ]:
# ⑨ 학습하기 (약 1분)
# [빈칸] 학습하는 함수 이름과 반복 횟수를 쓰세요.
history = model.____(X_train_scaler, y_train_hot,
                    epochs = ____, verbose = 2,
                    validation_data = (X_test_scaler, y_test_hot))

> **확인** : Epoch 1/10 … Epoch 10/10, 마지막 accuracy 약 0.99, val_accuracy 약 0.98

## ⑩ 성능 평가 — 그래프 (100쪽)
history에 기록된 에포크별 오차(loss)와 정확도(accuracy)를 그래프로 그린다. 검증 데이터 값은 `val_` 이 붙는다.

In [ ]:
# ⑩ 오차·정확도 그래프 그리기
# [빈칸] 검증(테스트) 데이터의 오차와 정확도 기록 이름을 쓰세요.
def HistoryShow(data):   # 오차 및 정확도를 그래프로 출력하는 함수
  plt.plot(data.history['loss'], label = '훈련 데이터 오차')
  plt.plot(data.history['________'], label = '테스트 데이터 오차')
  plt.plot(data.history['accuracy'], label = '훈련 데이터 정확도')
  plt.plot(data.history['____________'], label = '테스트 데이터 정확도')
  plt.legend( )
  plt.show( )
HistoryShow(history)

> **확인** : 오차 두 선은 0 쪽으로, 정확도 두 선은 1 쪽으로. 손실률은 0에 가까울수록, 정확도는 1에 가까울수록 좋다.

## ⑪ 딥러닝 정확도 (101쪽)
`model.evaluate( )` 는 [오차, 정확도]를 돌려준다 → `[1]` 이 정확도

In [ ]:
# ⑪ 훈련·테스트 데이터 정확도
# [빈칸] 케라스 모델을 평가하는 함수 이름을 쓰세요.
print('훈련 데이터 정확도: {:.3f}'.format(model.________(X_train_scaler, y_train_hot, verbose=0)[1]))
print('테스트 데이터 정확도: {:.3f}'.format(model.evaluate(X_test_scaler, y_test_hot, verbose=0)[1]))

> **확인** : 훈련 약 0.992, 테스트 약 0.976

## ⑫ 기계학습 정확도 (101쪽)
같은 데이터로 **의사결정트리**(83쪽)를 학습시켜 딥러닝과 비교한다. 사이킷런은 원-핫 인코딩이 필요 없어 `y_train` 을 그대로 쓴다.

In [ ]:
# ⑫ 의사결정트리로 같은 문제 풀어 비교하기
# [빈칸] 의사결정트리 분류 클래스 이름, 학습 함수 이름, 학습에 넣을 정답 변수를 쓰세요.
from sklearn.tree import ____________
tree = DecisionTreeClassifier(random_state = 0)   # 의사결정트리 생성
tree.____(X_train_scaler, _______)                  # 학습
print('훈련 데이터 정확도: {:.3f}'.format(tree.score(X_train_scaler, y_train)))
print('테스트 데이터 정확도: {:.3f}'.format(tree.score(X_test_scaler, y_test)))

> **확인** : 훈련 1.000, 테스트 약 0.878 → 훈련 데이터만 외운 **과적합**. 딥러닝(0.976)이 새 데이터에서 더 잘 맞힌다.

## 정리
- 딥러닝은 **특징 추출부터 학습까지 한 번에** 한다. (기계학습은 사람이 특징을 추출)
- 전처리: 차원 변환(784) → 정규화(÷255) → 원-핫 인코딩
- 모델: 입력 784 → 은닉 100(렐루) → 은닉 50(렐루) → 출력 10(소프트맥스), 손실함수 범주형 교차 엔트로피
- 평가: 딥러닝 0.976 vs 의사결정트리 0.878 (과적합)
